In [1]:
import torch
print(torch.__version__)
import pandas as pd
# pd.set_option('display.max_colwidth', None)
from time import time
import torch.nn as nn
from utils import *
from models import *
import torch.optim as optim

gpu_id = 0
source_domain = "movie"
target_domain = "music"
# 20 or 50 or 80
ratio = "80"

2.5.1


In [ ]:
def start(gpu_id, source_domain, target_domain, ratio):
    device = torch.device(f'cuda:{gpu_id}' if torch.cuda.is_available() and gpu_id < torch.cuda.device_count() else 'cpu')
    pre_weight_save = mf_weight[f"{source_domain}_100"]
    tpre_weight_save = mf_weight[f"{source_domain}_{target_domain}_{ratio}"]
    weight_save = emcdr_weight[f"{source_domain}_{target_domain}_{ratio}"]
    pt_path = os.path.join(train_data[f"{source_domain}_{target_domain}_save"], f"{ratio}train.pt")
    # target->source dict
    overlap_tgt2src = overlap(os.path.join(overlap_save, f"{target_domain}-{source_domain}-overlap.txt"))
    ttrain_uid, ttrain_iid, ttrain_rates, ttest_uid, ttest_iid, ttest_rates, n_user, n_item, tn_user, tn_item\
    = load_main_pt(pt_path, device)

    # model innitialization
    # Pre train
    lightgcn_source = MFEncoder(n_user, n_item, EMBBEDDING_DIM)
    lightgcn_source = lightgcn_source.to(device)
    lightgcn_source.load_state_dict(torch.load(pre_weight_save, weights_only=True))
    lightgcn_target = MFEncoder(tn_user, tn_item, EMBBEDDING_DIM)
    lightgcn_target = lightgcn_target.to(device)
    lightgcn_target.load_state_dict(torch.load(tpre_weight_save, weights_only=True))
    for p in lightgcn_source.parameters(): p.requires_grad = False
    for p in lightgcn_target.parameters(): p.requires_grad = False
    # train
    emcdr = EMCDR(EMBBEDDING_DIM)
    emcdr = emcdr.to(device)
    # Optimizer
    optimizer_pre = optim.Adam(
        filter(lambda p: p.requires_grad, emcdr.parameters()),
        lr=EMCDR_LR,
        # weight_decay=1e-5
    )
    sigmoid = nn.Sigmoid()

    # Train Process
    train_losses=[]
    s = time()
    for iter in range(EMCDR_ITERATIONS):
        emcdr.train()
        # We don't need items or ratings to train EMCDR! We just need overlapping users.
        for batch_uid, _, _ in mini_batch_iterator(ttrain_uid, ttrain_iid, ttrain_rates, batch_size=BATCH_SIZE):

            user_indices = batch_uid.to(device) # Target domain IDs
            
            # get source ids
            retrieved_values = [overlap_tgt2src[key] for key in user_indices.tolist()]

            # 1. Get Source Embeddings
            s_users, _ = lightgcn_source()
            s_users_batch = s_users[retrieved_values]

            # 2. Get Target Embeddings (Ground Truth for Mapper)
            t_users, _ = lightgcn_target()
            t_users_batch = t_users[user_indices].detach() # Important: detach to freeze!

            # 3. Map Source to Target
            mapped_users = emcdr(s_users_batch)

            # 4. Standard EMCDR Loss: MSE between Mapped User and Real Target User
            train_loss = MSELOSS(mapped_users, t_users_batch)

            optimizer_pre.zero_grad()
            train_loss.backward()
            optimizer_pre.step()

        if iter % ITERS_PER_EVAL == 0:
            emcdr.eval()
            with torch.no_grad():

                train_losses.append(train_loss.item())
            print(f"[Iteration {iter}/{ITERATIONS}] train_loss: {round(train_loss.item(), 5)}")

    end = time()
    print(f"Costing {end-s}s for training {ITERATIONS} iterations")

    torch.save(emcdr.state_dict(), weight_save)

    # test
    # Evaluate on overlapping users only
    current_idx = 0
    concatenated = torch.tensor([], dtype=torch.float32, device=device)
    concatenated_sig = torch.tensor([], dtype=torch.float32, device=device)
    emcdr.eval()
    with torch.no_grad():
        while True:
            batch_uid, batch_iid, batch_rates, current_idx = sample_mini_batch_sequential(
                ttest_uid, ttest_iid, ttest_rates,
                batch_size=BATCH_SIZE, current_idx=current_idx
            )
            if batch_uid is None:
                break  # reached the end

            retrieved_values = []
            for key in batch_uid.tolist():
                retrieved_values.append(overlap_tgt2src[key])
            # Source domain: all users
            s_users, _ = lightgcn_source()
            s_users = s_users[retrieved_values]
            final_user = emcdr(s_users)
            _, t_items = lightgcn_target()
            t_items = t_items[batch_iid]

            predicted_ratings_sig = sigmoid(torch.sum(final_user * t_items, dim=1))
            predicted_ratings = torch.sum(final_user * t_items, dim=1)
    
            concatenated = torch.cat([concatenated, predicted_ratings])
            concatenated_sig = torch.cat([concatenated_sig, predicted_ratings_sig])

    # rating calculation
    concatenated = (concatenated * RATES_MULTIPLY) + RATES_ADD
    concatenated = concatenated.reshape(len(ttest_rates))
    concatenated_sig = (concatenated_sig * RATES_MULTIPLY) + RATES_ADD
    concatenated_sig = concatenated_sig.reshape(len(ttest_rates))
    print(f"{source_domain}-{target_domain}_{ratio} Test Result:")
    print(f"RMSE: {float(rmse(concatenated_sig, ttest_rates))}")
    print(f"MAE: {float(mae(concatenated_sig, ttest_rates))}")
    print(f"R2: {float(r2(concatenated_sig, ttest_rates))}")

Train

In [6]:
# Example Usage
# start(gpu_id, source_domain, target_domain, ratio)

Costing 21.267564296722412s for training 200 iterations
movie-music_20 Test Result:
tensor(1.7957, device='cuda:0')
tensor(1.5196, device='cuda:0')
Costing 56.71116900444031s for training 200 iterations
movie-music_50 Test Result:
tensor(1.5512, device='cuda:0')
tensor(1.3107, device='cuda:0')
Costing 93.17093253135681s for training 200 iterations
movie-music_80 Test Result:
tensor(1.4277, device='cuda:0')
tensor(1.2131, device='cuda:0')

Costing 28.042181253433228s for training 200 iterations
music-movie_20 Test Result:
tensor(1.3704, device='cuda:0')
tensor(1.1179, device='cuda:0')
Costing 71.44721341133118s for training 200 iterations
music-movie_50 Test Result:
tensor(1.2807, device='cuda:0')
tensor(1.0401, device='cuda:0')
Costing 110.70814919471741s for training 200 iterations
music-movie_80 Test Result:
tensor(1.2236, device='cuda:0')
tensor(0.9806, device='cuda:0')

Costing 39.113728523254395s for training 200 iterations
book-movie_20 Test Result:
tensor(1.3814, device='cuda:0')
tensor(1.1396, device='cuda:0')
Costing 101.14235353469849s for training 200 iterations
book-movie_50 Test Result:
tensor(1.2778, device='cuda:0')
tensor(1.0442, device='cuda:0')
Costing 161.1933672428131s for training 200 iterations
book-movie_80 Test Result:
tensor(1.2499, device='cuda:0')
tensor(1.0057, device='cuda:0')

In [7]:
# iters = [iter * ITERS_PER_EVAL for iter in range(len(train_losses))]
# plt.plot(iters, train_losses, label='train')
# plt.xlabel('iteration')
# plt.ylabel('loss')
# plt.title('training and validation loss curves')
# plt.legend()
# plt.show()

In [8]:
# emcdr.load_state_dict(torch.load(weight_save + weight_name, weights_only=True))

Visualize

In [9]:
# indices = torch.randint(0, len(ttrain_uid), (1000,)).to(device)
# vz_ttuid = ttrain_uid[indices]
# vz_ttiid = ttrain_iid[indices]
# vz_tsuid = []
# for key in vz_ttuid.tolist():
#     vz_tsuid.append(overlap_tgt2src[key])
# tu, tv = lightgcn_target()
# su, sv = lightgcn_source()
# su = su[vz_tsuid]
# tu = tu[vz_ttuid]
# tv = tv[vz_ttiid]
# final_user = emcdr(su)

# tsne2d(su, tu, tv, final_user, labels=["su", "tu", "tv", "final_user"])

In [10]:
# tsne2d(su, tu, tv, labels=["su", "tu", "tv"])

music-movie 20%
rmse: 1.43895
mae: 1.11238
[Iteration 490/500] train_loss: 0.05146
Costing 79.5446937084198s for training 500 iterations
music-movie 50%
rmse: 1.38841
mae: 1.08673
[Iteration 490/500] train_loss: 0.06797
Costing 209.11017560958862s for training 500 iterations
music-movie 80%
rmse: 1.33815
mae: 1.05179
[Iteration 490/500] train_loss: 0.06851
Costing 320.20343136787415s for training 500 iterations

movie-music 20%
rmse: 1.72225
mae: 1.3427
Costing 81s for training 500 iterations
movie-music 50%
rmse: 1.6358
mae: 1.3173
Costing 264s for training 500 iterations
movie-music 80%
rmse: 1.57048
mae: 1.29121
Costing 417s for training 500 iterations

book-movie 20%
rmse: 1.44705
mae: 1.10528
[Iteration 490/500] train_loss: 0.06857
Costing 128.45861387252808s for training 500 iterations
book-movie 50%
rmse: 1.36322
mae: 1.05352
[Iteration 490/500] train_loss: 0.07791
Costing 328.1151375770569s for training 500 iterations
book-movie 80%
rmse: 1.34537
mae: 1.04023
[Iteration 490/500] train_loss: 0.07543
Costing 548.1038572788239s for training 500 iterations